# Wine Classification Notebook Workflow

This notebook is the full, notebook-only workflow for the project.

- Loads the wine CSV from `data/wine.csv`
- Trains and evaluates a logistic regression model
- Validates outputs with simple checks
- Plots and saves a confusion matrix


## 1. Environment setup

Run this cell first if needed.

In [ ]:
%pip install -q pandas numpy scikit-learn matplotlib seaborn

## 2. Load data and define helpers

The dataset lives in `data/wine.csv`.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

PROJECT_ROOT = Path('.')
DATA_PATH = PROJECT_ROOT / 'data' / 'wine.csv'
OUTPUT_PATH = PROJECT_ROOT / 'wine_confusion_matrix.png'
TARGET_COLUMN = 'target'
CLASS_NAMES = ['Cultivar 1', 'Cultivar 2', 'Cultivar 3']
TEST_SIZE = 0.2
RANDOM_STATE = 42

def load_data(path):
    return pd.read_csv(path)

def preprocess(df):
    X = df.drop(columns=[TARGET_COLUMN])
    y = df[TARGET_COLUMN]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
    )

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    return X_train_scaled, X_test_scaled, y_train, y_test

def train_model(X_train, y_train):
    model = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
    model.fit(X_train, y_train)
    return model

def evaluate_model(model, X_test, y_test):
    y_pred = model.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    report = classification_report(y_test, y_pred, target_names=CLASS_NAMES)
    cm = confusion_matrix(y_test, y_pred)
    return accuracy, report, cm, y_pred

def plot_confusion_matrix(cm, output_path):
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
    plt.title('Confusion Matrix — Wine Classification')
    plt.xlabel('Predicted Label')
    plt.ylabel('True Label')
    plt.tight_layout()
    plt.savefig(output_path)
    plt.show()

## 3. Train and test the model

Run the full workflow from load to evaluation.

In [ ]:
df = load_data(DATA_PATH)
print(f'Loaded wine data: {df.shape[0]} rows and {df.shape[1] - 1} features')

X_train, X_test, y_train, y_test = preprocess(df)
model = train_model(X_train, y_train)
accuracy, report, cm, y_pred = evaluate_model(model, X_test, y_test)

print(f'Train size: {len(y_train)}')
print(f'Test size: {len(y_test)}')
print(f'Accuracy: {accuracy:.4f}')
print('\nClassification report:')
print(report)

## 4. Validate and finish

Simple checks plus the final confusion matrix.

In [ ]:
assert df.shape == (178, 14)
assert df[TARGET_COLUMN].isnull().sum() == 0
assert set(df[TARGET_COLUMN].unique()) == {1, 2, 3}
assert len(y_train) == 142
assert len(y_test) == 36
assert accuracy > 0.90
assert cm.shape == (3, 3)

plot_confusion_matrix(cm, OUTPUT_PATH)
print(f'Confusion matrix saved to {OUTPUT_PATH}')